1) добавить таблицу с монтажом
2) запустить финальный обход
3) посмотреть на результат
4) придумать план

In [1]:
import sqlite3
import pandas as pd
import os
from tqdm import tqdm

In [2]:
con = sqlite3.connect('films.db')  # подключение
cur = con.cursor()  # курсор

## Метаданные

In [3]:
files_metadata = pd.read_csv('films_meta.tsv', sep='\t', header=0)
files_metadata['filename'] = files_metadata['eng title'].str.replace(":", "").str.replace(r"/", "").str.replace(" ", "_")
files_metadata.rename(columns={'rus title': 'rus_title', 'eng title': 'eng_title'}, inplace=True)
files_metadata['mark'] = pd.to_numeric(files_metadata['mark'].str.replace(',', '.'))
files_metadata.head(3)

,year,genre,eng_title,rus_title,imdb,mark,rated,released,runtime,genres,directors,writers,actors,awards,json,path,filename
0,1990,action,The Hunt for Red October,Охота за Красным Октябрем,https://www.imdb.com/title/tt0099810/,7.6,PG,02 Mar 1990,135 min,"Action, Adventure, Thriller",John McTiernan,"Tom Clancy, Larry Ferguson, Donald E. Stewart","Sean Connery, Alec Baldwin, Scott Glenn",Won 1 Oscar. 3 wins & 9 nominations total,"{'Title': 'The Hunt for Red October', 'Year': ...",/media/boris/HD-E1/USA1990/1990/Ohota_za_Krasn...,The_Hunt_for_Red_October
1,1990,action,Total Recall,Вспомнить всё,https://www.imdb.com/title/tt0100802/,7.5,R,01 Jun 1990,113 min,"Action, Adventure, Sci-Fi",Paul Verhoeven,"Philip K. Dick, Ronald Shusett, Dan O'Bannon","Arnold Schwarzenegger, Sharon Stone, Michael I...",Nominated for 2 Oscars. 7 wins & 16 nomination...,"{'Title': 'Total Recall', 'Year': '1990', 'Rat...",/media/boris/HD-E1/USA1990/1990/Total.Recall.1...,Total_Recall
2,1990,action,Die Hard 2,Крепкий орешек 2,https://www.imdb.com/title/tt0099423/,7.2,R,03 Jul 1990,124 min,"Action, Thriller",Renny Harlin,"Steven E. de Souza, Doug Richardson, Walter Wager","Bruce Willis, William Atherton, Bonnie Bedelia",1 win & 1 nomination total,"{'Title': 'Die Hard 2', 'Year': '1990', 'Rated...",/media/boris/HD-E1/USA1990/1990/Die.Hard.2.199...,Die_Hard_2


In [4]:
films_data = pd.read_csv('films_data_without_titles.csv', header=0)
films_data['filename'] = films_data['filename'].str.replace(' ', '_')
metadata_2 = films_data.groupby(by='filename', as_index=False)[['height', 'width', 'frame_count', 'shot_count', 'film_start', 'film_end']].max()
metadata_2.head(3)

,filename,height,width,frame_count,shot_count,film_start,film_end
0,10_Things_I_Hate_About_You,384,704,130987,813,1891,132877
1,A_Bronx_Tale,512,688,156357,1730,3855,160211
2,A_Few_Good_Men,304,720,192462,1462,713,193174


In [5]:
files_metadata = pd.merge(files_metadata, metadata_2, on='filename', how='left')
files_metadata.head(3)

,year,genre,eng_title,rus_title,imdb,mark,rated,released,runtime,genres,...,awards,json,path,filename,height,width,frame_count,shot_count,film_start,film_end
0,1990,action,The Hunt for Red October,Охота за Красным Октябрем,https://www.imdb.com/title/tt0099810/,7.6,PG,02 Mar 1990,135 min,"Action, Adventure, Thriller",...,Won 1 Oscar. 3 wins & 9 nominations total,"{'Title': 'The Hunt for Red October', 'Year': ...",/media/boris/HD-E1/USA1990/1990/Ohota_za_Krasn...,The_Hunt_for_Red_October,304,720,188041,1151,1617,189657
1,1990,action,Total Recall,Вспомнить всё,https://www.imdb.com/title/tt0100802/,7.5,R,01 Jun 1990,113 min,"Action, Adventure, Sci-Fi",...,Nominated for 2 Oscars. 7 wins & 16 nomination...,"{'Title': 'Total Recall', 'Year': '1990', 'Rat...",/media/boris/HD-E1/USA1990/1990/Total.Recall.1...,Total_Recall,384,704,157016,1561,908,157923
2,1990,action,Die Hard 2,Крепкий орешек 2,https://www.imdb.com/title/tt0099423/,7.2,R,03 Jul 1990,124 min,"Action, Thriller",...,1 win & 1 nomination total,"{'Title': 'Die Hard 2', 'Year': '1990', 'Rated...",/media/boris/HD-E1/USA1990/1990/Die.Hard.2.199...,Die_Hard_2,304,720,167616,2121,734,168349


In [6]:
files_metadata[files_metadata['frame_count'].isnull()]

,year,genre,eng_title,rus_title,imdb,mark,rated,released,runtime,genres,...,awards,json,path,filename,height,width,frame_count,shot_count,film_start,film_end


In [7]:
files_metadata['released'] = pd.to_datetime(files_metadata['released'])
files_metadata['runtime'] = files_metadata['runtime'].apply(lambda x: int(x[:-4]))

In [8]:
files_metadata.info()

<class 'pandas.DataFrame'>
RangeIndex: 298 entries, 0 to 297
Data columns (total 23 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   year         298 non-null    int64         
 1   genre        298 non-null    str           
 2   eng_title    298 non-null    str           
 3   rus_title    298 non-null    str           
 4   imdb         298 non-null    str           
 5   mark         298 non-null    float64       
 6   rated        297 non-null    str           
 7   released     298 non-null    datetime64[us]
 8   runtime      298 non-null    int64         
 9   genres       298 non-null    str           
 10  directors    298 non-null    str           
 11  writers      298 non-null    str           
 12  actors       298 non-null    str           
 13  awards       280 non-null    str           
 14  json         298 non-null    str           
 15  path         298 non-null    str           
 16  filename     298 no

In [9]:
files_metadata = files_metadata[['filename', 'year', 'genre', 'eng_title', 'rus_title',
                                 'height', 'width', 'frame_count', 'shot_count', 'film_start',
                                 'film_end', 'imdb', 'mark', 'rated', 'released', 'runtime', 'genres',
                                 'directors', 'writers', 'actors', 'awards']]
files_metadata = files_metadata.rename(columns={'filename':'title'})

In [10]:
files_metadata = files_metadata.reset_index()
files_metadata.sample(2)

,index,title,year,genre,eng_title,rus_title,height,width,frame_count,shot_count,...,imdb,mark,rated,released,runtime,genres,directors,writers,actors,awards
81,81,Universal_Soldier,1992,sci-fi,Universal Soldier,Универсальный солдат,304,704,140546,1697,...,https://www.imdb.com/title/tt0105698/,6.0,R,1992-07-10,102,"Action, Sci-Fi",Roland Emmerich,"Richard Rothstein, Christopher Leitch, Dean De...","Jean-Claude Van Damme, Dolph Lundgren, Ally Wa...",NaN
173,173,"Cry,_the_Beloved_Country",1995,thriller,"Cry, the Beloved Country",Два цвета времени,400,720,143267,1124,...,https://www.imdb.com/title/tt0112749/,6.8,PG-13,1995-12-15,106,"Drama, Thriller",Darrell Roodt,"Ronald Harwood, Alan Paton","Richard Harris, James Earl Jones, Vusi Kunene",1 win & 4 nominations


In [11]:
cur.execute("DROP TABLE IF EXISTS metadata;")

In [12]:
files_metadata.to_sql('metadata', con, if_exists='append', index=False)

298

## Планы фильма

In [13]:
films_data

,shot_index,start,end,filename,length,time_length,timecode,height,width,frame_count,shot_count,film_start,film_end,part,part_of_6,part_of_9
0,0,1891,1957,10_Things_I_Hate_About_You,66,2.750000,0:01:18,384,704,130987,813,1891,132877,1,1,1
1,1,1958,2049,10_Things_I_Hate_About_You,91,3.791667,0:01:21,384,704,130987,813,1891,132877,1,1,1
2,2,2050,2112,10_Things_I_Hate_About_You,62,2.583333,0:01:25,384,704,130987,813,1891,132877,1,1,1
3,3,2113,2555,10_Things_I_Hate_About_You,442,18.416667,0:01:28,384,704,130987,813,1891,132877,1,1,1
4,4,2556,2781,10_Things_I_Hate_About_You,225,9.375000,0:01:46,384,704,130987,813,1891,132877,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
392083,521,146173,146323,Xi_yan,150,6.250000,1:41:30,384,704,145496,526,1539,147034,3,6,9
392084,522,146324,146505,Xi_yan,181,7.541667,1:41:36,384,704,145496,526,1539,147034,3,6,9
392085,523,146506,146735,Xi_yan,229,9.541667,1:41:44,384,704,145496,526,1539,147034,3,6,9
392086,524,146736,146854,Xi_yan,118,4.916667,1:41:54,384,704,145496,526,1539,147034,3,6,9


In [14]:
films_data = films_data[['filename', 'shot_index', 'start', 'end', 'length', 'time_length', 'timecode', 'part', 'part_of_6', 'part_of_9']]
films_data = films_data.rename(columns={'filename':'title'})
films_data.head()

,title,shot_index,start,end,length,time_length,timecode,part,part_of_6,part_of_9
0,10_Things_I_Hate_About_You,0,1891,1957,66,2.750000,0:01:18,1,1,1
1,10_Things_I_Hate_About_You,1,1958,2049,91,3.791667,0:01:21,1,1,1
2,10_Things_I_Hate_About_You,2,2050,2112,62,2.583333,0:01:25,1,1,1
3,10_Things_I_Hate_About_You,3,2113,2555,442,18.416667,0:01:28,1,1,1
4,10_Things_I_Hate_About_You,4,2556,2781,225,9.375000,0:01:46,1,1,1


In [15]:
cur.execute("DROP TABLE IF EXISTS dvt_data;")

In [16]:
cur.execute('''
    CREATE TABLE IF NOT EXISTS dvt_data (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        title TEXT,
        shot_index INTEGER,
        start INTEGER,
        end INTEGER,
        length INTEGER,
        time_length REAL,
        timecode TEXT,
        part INTEGER,
        part_of_6 INTEGER,
        part_of_9 INTEGER
    )
''')

In [17]:
films_data.to_sql('dvt_data', con, if_exists='append', index=False)

392088

## Покадровые метрики

In [18]:
cur.execute("DROP TABLE IF EXISTS frames;")

In [19]:
def find_length(data, frame):

    if frame < data.iloc[0]['start']:
        return -1
    if frame > data.iloc[-1]['end']:
        return -2
    
    low = 0
    high = len(data) - 1
    
    while low <= high:
        middle = (low + high) // 2
        if frame < data.iloc[middle]['start']:
            high = middle - 1
        elif frame > data.iloc[middle]['end']:
            low = middle + 1
        else:
            return data.iloc[middle]['shot_index']
    
    return -3

In [20]:
bright_df = pd.read_csv('brightness_data.csv', header=0)

In [21]:
bright_df.sample(3)

,year,genre,filename,nframe,brightness
1289473,1996,comedy,Box_of_Moonlight,97584,105.984641
1387679,1996,horror,The_Frighteners,54528,76.513503
674018,1993,crime,In_the_Line_of_Fire,67872,64.557580


In [22]:
# tsv_all = pd.DataFrame()
for filename in tqdm(os.listdir('tsv')):
    
    title = filename.split('-')[-1].replace("'",'_').replace('_.', '.')[:-4]
    title_2 = filename.split('-')[-1].replace('_.', '.')[:-4]
    tsv_df = pd.read_csv(f'tsv/{filename}', sep='\t', header=0)
    tsv_df = pd.merge(tsv_df, bright_df[bright_df['filename'] == title][['nframe', 'brightness']], on='nframe', how='left')
    
    dvt_data = films_data[films_data['title'] == title_2]
    if dvt_data.shape[0] == 0:
        print(filename)
    tsv_df['shot_index'] = tsv_df['nframe'].apply(lambda x: find_length(dvt_data, x))
    tsv_df['title'] = title_2
    tsv_df = tsv_df.drop(['h', 'm', 's'], axis=1)

    # tsv_all = pd.concat([tsv_all, tsv_df], axis=0, ignore_index=False)
    tsv_df.to_sql('frames', con, if_exists='append', index=False)

100%|████████████████████████████████████████████████████████████████████████████████| 298/298 [43:18<00:00,  8.72s/it]


In [23]:
tsv_df.sample(5)

,nframe,faces,fsquare,neyes,nobjects,brightness,shot_index,title
4945,118680,0,0.000,0,207,127.337889,1164,Three_Kings
419,10056,0,0.000,0,468,138.136866,68,Three_Kings
4529,108696,1,0.038,0,438,83.027170,1074,Three_Kings
1757,42168,0,0.000,1,379,185.368548,403,Three_Kings
6363,152712,0,0.000,2,274,96.223063,1511,Three_Kings


In [24]:
cur.execute("""SELECT * FROM frames""")

In [25]:
cur.fetchone()

(0, 0, 0.0, 0, 0, 0.0, -1, 'Die_Hard_2')

## Сохраняем изменения и закрываем базу данных

In [26]:
# create_table_sql = """
# CREATE TABLE IF NOT EXISTS frames AS
# SELECT 
#     m.id AS movie_id,  -- Берем id из таблицы метаданных
#     shot_index, nframe, h, m, s, faces, fsquare, neyes, nobjects
# FROM 
#     temp_frames t
# LEFT JOIN 
#     metadata m ON t.title = m.filename; -- Соединяем по названию
# """

# cur.execute(create_table_sql)

In [27]:
# cur.execute("DROP TABLE IF EXISTS temp_frames;")

In [28]:
con.commit()

In [29]:
con.close()

## Неактуальное

In [30]:
# con = sqlite3.connect('films.db')  # подключение
# cur = con.cursor()  # курсор

In [31]:
# sql_query = "SELECT shot_index, start, end, time_length, part, frame_count, shot_count, year, genre, rus_title FROM dvt_data AS dd JOIN metadata AS md ON dd.movie_id = md.id"

# df = pd.read_sql(sql_query, con)
# df.to_csv("db_shot.csv", index=False)

In [32]:
# sql_query = "SELECT nframe, h, m, s, faces, fsquare, neyes, nobjects, f.shot_index, shot_count, frame_count, time_length, part, year, genre, rus_title FROM frames AS f JOIN metadata AS md ON f.movie_id = md.id JOIN dvt_data AS dd ON f.movie_id = dd.movie_id AND f.shot_index = dd.shot_index"

# df = pd.read_sql(sql_query, con)
# df.to_csv("db_frames.csv", index=False)

In [33]:
# cur.executemany("INSERT INTO frames VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)", data)

In [34]:
def find_movie_id(row):
    cur.execute(f'''SELECT id FROM metadata WHERE filename = "{row['filename']}";''')
    return cur.fetchone()[0]

films_data['movie_id'] = films_data.apply(find_movie_id, axis=1)

KeyError: 'filename'

In [ ]:
films_data = films_data.drop('filename', axis = 'columns')
films_data.head(3)

In [ ]:
cur.execute('''
    CREATE TABLE IF NOT EXISTS metadata (
        id INTEGER,
        filename TEXT PRIMARY KEY,
        year INTEGER, 
        genre TEXT NOT NULL,
        eng_title TEXT,
        rus_title TEXT,
        frame_count INTEGER,
        shot_count INTEGER,
        film_start INTEGER,
        film_end INTEGER,
        imdb TEXT, 
        mark REAL,
        rated TEXT,
        released TEXT,
        runtime INTEGER,
        genres TEXT,
        directors TEXT,
        writers TEXT,
        actors TEXT,
        awards TEXT
    )
''')